In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Đọc dữ liệu sạch
df = pd.read_csv('../data/processed/hotel_bookings_clean.csv')

# Ép kiểu object về string đồng nhất
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype(str)

X = df.drop(columns=['is_canceled'])
y = df['is_canceled']

# Phiên bản A: Giữ đầy đủ biến (Bao gồm biến nhạy cảm 'country')
X_A = X.copy()

# Phiên bản B: Loại bỏ biến nhạy cảm 'country' để đảm bảo tính đạo đức/công bằng
X_B = X.drop(columns=['country'])

# Chia tập Train/Test cho 2 phiên bản
X_train_A, X_test_A, y_train_A, y_test_A = train_test_split(X_A, y, test_size=0.2, random_state=42, stratify=y)
X_train_B, X_test_B, y_train_B, y_test_B = train_test_split(X_B, y, test_size=0.2, random_state=42, stratify=y)

print("Đã chuẩn bị xong dữ liệu cho Phiên bản A (Có country) và Phiên bản B (Loại bỏ country)!")

Đã chuẩn bị xong dữ liệu cho Phiên bản A (Có country) và Phiên bản B (Loại bỏ country)!


/tmp/ipykernel_6568/2545244839.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:


In [2]:
def build_pipeline(X_data):
    num_cols = X_data.select_dtypes(include=['int64', 'float64']).columns.tolist()
    cat_cols = X_data.select_dtypes(include=['object']).columns.tolist()
    
    prep = ColumnTransformer(transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ])
    
    return Pipeline([
        ('prep', prep),
        ('model', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
    ])

# Huấn luyện Mô hình A
model_A = build_pipeline(X_train_A)
model_A.fit(X_train_A, y_train_A)
auc_A = roc_auc_score(y_test_A, model_A.predict_proba(X_test_A)[:, 1])

# Huấn luyện Mô hình B
model_B = build_pipeline(X_train_B)
model_B.fit(X_train_B, y_train_B)
auc_B = roc_auc_score(y_test_B, model_B.predict_proba(X_test_B)[:, 1])

print("--- KẾT QUẢ ĐÁNH GIÁ TÍNH CÔNG BẰNG & ĐẠO ĐỨC (CLO7) ---")
print(f"Phiên bản A (Có biến 'country'):      ROC-AUC = {auc_A:.4f}")
print(f"Phiên bản B (Loại bỏ biến 'country'): ROC-AUC = {auc_B:.4f}")
print(f"-> Mức độ chênh lệch hiệu năng:       {abs(auc_A - auc_B)*100:.2f}%")

/tmp/ipykernel_6568/3001241866.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_data.select_dtypes(include=['object']).columns.tolist()
/tmp/ipykernel_6568/3001241866.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes fo

--- KẾT QUẢ ĐÁNH GIÁ TÍNH CÔNG BẰNG & ĐẠO ĐỨC (CLO7) ---
Phiên bản A (Có biến 'country'):      ROC-AUC = 0.9591
Phiên bản B (Loại bỏ biến 'country'): ROC-AUC = 0.9384
-> Mức độ chênh lệch hiệu năng:       2.07%


In [3]:
print("--- LẬP LUẬN ĐẠO ĐỨC AI (ĐƯA VÀO BÁO CÁO CỦA NHÓM) ---")
print("1. Việc dùng biến 'country' có thể dẫn tới việc phân biệt đối xử (yêu cầu khách quốc tịch nhất định phải cọc cao hơn).")
print(f"2. Loại bỏ biến 'country' chỉ làm giảm {abs(auc_A - auc_B)*100:.2f}% điểm ROC-AUC của mô hình.")
print("3. Khuyến nghị: Doanh nghiệp nên sử dụng Mô hình B để đảm bảo tính công bằng và uy tín thương hiệu.")

--- LẬP LUẬN ĐẠO ĐỨC AI (ĐƯA VÀO BÁO CÁO CỦA NHÓM) ---
1. Việc dùng biến 'country' có thể dẫn tới việc phân biệt đối xử (yêu cầu khách quốc tịch nhất định phải cọc cao hơn).
2. Loại bỏ biến 'country' chỉ làm giảm 2.07% điểm ROC-AUC của mô hình.
3. Khuyến nghị: Doanh nghiệp nên sử dụng Mô hình B để đảm bảo tính công bằng và uy tín thương hiệu.
